In [5]:
# ============================================
# SEARCH QUERY SPELLING CORRECTOR
# BIRKBECK SPELLING ERROR CORPUS
# ============================================

import pandas as pd
import numpy as np
import re


# ============================================
# 1. LOAD BIRKBECK CORPUS
# ============================================

file_path = "birkbeck.dat.txt"

data = []

current_correct_word = None

with open(file_path, "r", encoding="latin-1") as file:

    for line in file:

        line = line.strip()

        # Skip empty lines
        if not line:
            continue

        # Correct word starts with $
        if line.startswith("$"):

            current_correct_word = line[1:].lower()

        else:

            # This is a misspelled word
            if current_correct_word is not None:

                incorrect_word = line.lower()

                data.append(
                    [incorrect_word, current_correct_word]
                )


# ============================================
# 2. CREATE DATAFRAME
# ============================================

corpus = pd.DataFrame(
    data,
    columns=["incorrect", "correct"]
)


print("Corpus loaded successfully!")

print("Number of spelling pairs:", len(corpus))

print("\nFirst 10 records:")
print(corpus.head(10))


# ============================================
# 3. BUILD CORRECT WORD VOCABULARY
# ============================================

vocabulary = set(corpus["correct"])

print("\nVocabulary size:", len(vocabulary))


# ============================================
# 4. EDIT DISTANCE FUNCTION
# ============================================

def edit_distance(word1, word2):

    rows = len(word1) + 1
    cols = len(word2) + 1

    matrix = np.zeros(
        (rows, cols),
        dtype=int
    )

    # First column
    for i in range(rows):
        matrix[i][0] = i

    # First row
    for j in range(cols):
        matrix[0][j] = j

    # Calculate edit distance
    for i in range(1, rows):

        for j in range(1, cols):

            if word1[i - 1] == word2[j - 1]:
                cost = 0
            else:
                cost = 1

            matrix[i][j] = min(

                matrix[i - 1][j] + 1,

                matrix[i][j - 1] + 1,

                matrix[i - 1][j - 1] + cost
            )

    return matrix[rows - 1][cols - 1]


# ============================================
# 5. FIND CLOSEST WORD
# ============================================

def correct_word(word):

    word = word.lower()

    # If word is already correct
    if word in vocabulary:

        return word, 0

    best_word = word
    best_distance = float("inf")

    for candidate in vocabulary:

        distance = edit_distance(
            word,
            candidate
        )

        if distance < best_distance:

            best_distance = distance
            best_word = candidate

    return best_word, best_distance


# ============================================
# 6. CORRECT SEARCH QUERY
# ============================================

def correct_query(query):

    # Convert query to lowercase
    query = query.lower()

    # Tokenize query
    words = re.findall(
        r"[a-zA-Z]+",
        query
    )

    corrected_words = []

    incorrect_words = []

    suggestions = []

    # Check every word
    for word in words:

        # If word is already correct
        if word in vocabulary:

            corrected_words.append(word)

        else:

            corrected_word, distance = correct_word(word)

            corrected_words.append(
                corrected_word
            )

            incorrect_words.append(word)

            suggestions.append(
                (
                    word,
                    corrected_word,
                    distance
                )
            )

    # Create final corrected query
    corrected_query = " ".join(
        corrected_words
    )

    return (
        words,
        incorrect_words,
        suggestions,
        corrected_query
    )


# ============================================
# 7. ACCEPT USER QUERY
# ============================================

query = input(
    "\nEnter your search query: "
)


# ============================================
# 8. CORRECT QUERY
# ============================================

words, incorrect_words, suggestions, corrected_query = \
    correct_query(query)


# ============================================
# 9. DISPLAY RESULTS
# ============================================

print("\n============================================")
print("SEARCH QUERY SPELLING CORRECTOR")
print("============================================")


# Original query
print("\nOriginal Query:")
print(query)


# Tokenized query
print("\nTokenized Query:")
print(words)


# Incorrect words
print("\nIncorrect Words:")

if len(incorrect_words) == 0:

    print("No spelling errors found.")

else:

    for word in incorrect_words:

        print(word)


# Suggested corrections
print("\nSuggested Corrections:")

if len(suggestions) == 0:

    print("No corrections required.")

else:

    for wrong, correct, distance in suggestions:

        print(
            wrong,
            "->",
            correct,
            "(Edit Distance:",
            distance,
            ")"
        )


# Final corrected query
print("\nFinal Corrected Query:")
print(corrected_query)


print("\n============================================")
print("PROCESS COMPLETED")
print("============================================")

Corpus loaded successfully!
Number of spelling pairs: 36133

First 10 records:
         incorrect       correct
0               ab        albert
1          ameraca       america
2          amercia       america
3         ameracan      american
4            apirl         april
5         austrain      austrian
6          badcock     badcock's
7  bechuarnia_land  bechuanaland
8         botuania      botswana
9         cambrige     cambridge

Vocabulary size: 6130



Enter your search query:  apr



SEARCH QUERY SPELLING CORRECTOR

Original Query:
apr

Tokenized Query:
['apr']

Incorrect Words:
apr

Suggested Corrections:
apr -> air (Edit Distance: 1 )

Final Corrected Query:
air

PROCESS COMPLETED
